In [1]:
import pandas as pd

In [2]:
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [6]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

NUM_LOCATIONS = 20
NUM_CATEGORIES = 10
NUM_MENU_ITEMS = 150
NUM_CUSTOMERS = 50000
NUM_ORDERS = 100000
NUM_ORDER_ITEMS_TARGET = 1000000
NUM_PROMOTIONS = 20
NUM_RATINGS = 100000
NUM_WASTAGE = 50000

cities = ["Karachi", "Lahore", "Islamabad", "Faisalabad", "Rawalpindi", "Multan", "Peshawar", "Quetta"]
channels = np.array(["Dine-in", "Takeaway", "Website", "App", "Third-Party Delivery"])
category_names = ["Starters", "Soups", "Salads", "Main Course", "Beverages", "Desserts", "Breads", "Rice & Biryani", "Fast Food", "BBQ"]

restaurants = []
for i in range(1, NUM_LOCATIONS + 1):
    restaurants.append({
        "Location_ID": i,
        "Location_Name": fake.company() + " - " + random.choice(cities),
        "City": random.choice(cities),
        "Address": fake.address().replace("\n", ", "),
        "Opening_Date": fake.date_between(start_date="-3y", end_date="-6m")
    })
df_restaurants = pd.DataFrame(restaurants)

df_categories = pd.DataFrame([{"Category_ID": i, "Category_Name": category_names[i - 1]} for i in range(1, NUM_CATEGORIES + 1)])

menu_items = []
for i in range(1, NUM_MENU_ITEMS + 1):
    cost = round(random.uniform(150, 900), 2)
    price = round(cost * random.uniform(1.3, 2.8), 2)
    menu_items.append({
        "Item_ID": i,
        "Item_Name": fake.unique.word().capitalize() + " " + random.choice(["Platter", "Special", "Deluxe", "Combo", "Classic"]),
        "Category_ID": random.randint(1, NUM_CATEGORIES),
        "Base_Price": price,
        "Cost": cost,
        "Description": fake.sentence(nb_words=8),
        "Is_Available": random.choices([1, 0], weights=[95, 5])[0]
    })
df_menu_items = pd.DataFrame(menu_items)
bad_price_idx = df_menu_items.sample(5, random_state=1).index
df_menu_items.loc[bad_price_idx, "Base_Price"] = -1
item_prices = df_menu_items["Base_Price"].values
item_prices_clean = np.where(item_prices < 0, df_menu_items["Cost"].values * 1.5, item_prices)

first_names = [fake.first_name() for _ in range(3000)]
last_names = [fake.last_name() for _ in range(3000)]
cust_first = np.random.choice(first_names, NUM_CUSTOMERS)
cust_last = np.random.choice(last_names, NUM_CUSTOMERS)
cust_names = [f"{f} {l}" for f, l in zip(cust_first, cust_last)]
signup_days_ago = np.random.randint(0, 730, NUM_CUSTOMERS)
signup_dates = [datetime.now().date() - timedelta(days=int(d)) for d in signup_days_ago]
df_customers = pd.DataFrame({
    "Customer_ID": np.arange(1, NUM_CUSTOMERS + 1),
    "Customer_Name": cust_names,
    "City": np.random.choice(cities, NUM_CUSTOMERS),
    "Signup_Date": signup_dates,
    "Preferred_Channel": np.random.choice(channels, NUM_CUSTOMERS)
})
dup_customers = df_customers.sample(40, random_state=2)
df_customers = pd.concat([df_customers, dup_customers], ignore_index=True)
missing_name_idx = df_customers.sample(120, random_state=3).index
df_customers.loc[missing_name_idx, "Customer_Name"] = None

promo_types = ["Percentage Discount", "Flat Discount", "Buy1Get1", "Combo Deal"]
promotions = []
for i in range(1, NUM_PROMOTIONS + 1):
    start = fake.date_between(start_date="-1y", end_date="-30d")
    end = start + timedelta(days=random.randint(7, 45))
    promotions.append({
        "Promotion_ID": i,
        "Promotion_Name": random.choice(promo_types) + " " + str(i),
        "Discount_Percent": random.choice([10, 15, 20, 25, 30, 0]),
        "Start_Date": start,
        "End_Date": end,
        "Applicable_Category_ID": random.randint(1, NUM_CATEGORIES)
    })
df_promotions = pd.DataFrame(promotions)

order_start = datetime.now() - timedelta(days=365)
offsets_seconds = np.random.randint(0, 365 * 24 * 3600, NUM_ORDERS)
order_datetimes = [order_start + timedelta(seconds=int(s)) for s in offsets_seconds]
is_promo_mask = np.random.random(NUM_ORDERS) < 0.25
promo_ids = np.where(is_promo_mask, np.random.randint(1, NUM_PROMOTIONS + 1, NUM_ORDERS), 0)
promo_ids = promo_ids.astype(object)
promo_ids[promo_ids == 0] = None
df_orders = pd.DataFrame({
    "Order_ID": np.arange(1, NUM_ORDERS + 1),
    "Customer_ID": np.random.randint(1, NUM_CUSTOMERS + 1, NUM_ORDERS),
    "Location_ID": np.random.randint(1, NUM_LOCATIONS + 1, NUM_ORDERS),
    "Order_DateTime": order_datetimes,
    "Channel": np.random.choice(channels, NUM_ORDERS),
    "Promotion_ID": promo_ids,
    "Order_Status": np.random.choice(["Completed", "Cancelled"], NUM_ORDERS, p=[0.96, 0.04])
})
dup_orders = df_orders.sample(80, random_state=4)
df_orders = pd.concat([df_orders, dup_orders], ignore_index=True)
invalid_date_idx = df_orders.sample(30, random_state=5).index
df_orders.loc[invalid_date_idx, "Order_DateTime"] = pd.NaT

order_ids = df_orders["Order_ID"].values
num_orders = len(order_ids)
base_per_order = NUM_ORDER_ITEMS_TARGET // num_orders
remainder = NUM_ORDER_ITEMS_TARGET - base_per_order * num_orders
items_per_order = np.full(num_orders, base_per_order)
extra_idx = np.random.choice(num_orders, remainder, replace=False)
items_per_order[extra_idx] += 1

order_id_repeated = np.repeat(order_ids, items_per_order)
n = len(order_id_repeated)
item_ids = np.random.randint(1, NUM_MENU_ITEMS + 1, n)
quantities = np.random.choice([1, 2, 3], n, p=[0.7, 0.22, 0.08])
unit_prices = item_prices_clean[item_ids - 1]
discount_flags = np.random.random(n) < 0.2
discount_rate = np.where(discount_flags, np.random.choice([0.1, 0.15], n), 0)
discount_applied = np.round(unit_prices * quantities * discount_rate, 2)
df_order_items = pd.DataFrame({
    "Order_Item_ID": np.arange(1, n + 1),
    "Order_ID": order_id_repeated,
    "Item_ID": item_ids,
    "Quantity": quantities,
    "Unit_Price": unit_prices,
    "Discount_Applied": discount_applied
})
neg_qty_idx = df_order_items.sample(150, random_state=6).index
df_order_items.loc[neg_qty_idx, "Quantity"] = -1

pricing_history = []
ph_counter = 1
base_prices = dict(zip(df_menu_items["Item_ID"], df_menu_items["Base_Price"]))
for item_id in df_menu_items["Item_ID"]:
    num_changes = random.randint(1, 4)
    current_price = base_prices[item_id]
    change_date = datetime.now() - timedelta(days=365)
    for _ in range(num_changes):
        change_date += timedelta(days=random.randint(30, 120))
        current_price = round(current_price * random.uniform(0.9, 1.15), 2)
        pricing_history.append({"Pricing_ID": ph_counter, "Item_ID": item_id, "Price": current_price, "Effective_Date": change_date})
        ph_counter += 1
df_pricing_history = pd.DataFrame(pricing_history)

completed_order_ids = df_orders.loc[df_orders["Order_Status"] == "Completed", "Order_ID"].values
order_cust_map = dict(zip(df_orders["Order_ID"], df_orders["Customer_ID"]))
order_dt_map = dict(zip(df_orders["Order_ID"], df_orders["Order_DateTime"]))
rating_order_ids = np.random.choice(completed_order_ids, NUM_RATINGS)
rating_customers = [order_cust_map[o] for o in rating_order_ids]
rating_stars = np.random.choice([1, 2, 3, 4, 5], NUM_RATINGS, p=[0.03, 0.05, 0.12, 0.35, 0.45])
rating_dates = []
for o in rating_order_ids:
    dt = order_dt_map[o]
    if pd.notna(dt):
        rating_dates.append(dt + timedelta(hours=int(np.random.randint(1, 48))))
    else:
        rating_dates.append(None)
df_ratings = pd.DataFrame({
    "Rating_ID": np.arange(1, NUM_RATINGS + 1),
    "Order_ID": rating_order_ids,
    "Item_ID": np.random.randint(1, NUM_MENU_ITEMS + 1, NUM_RATINGS),
    "Customer_ID": rating_customers,
    "Stars": rating_stars,
    "Rating_Date": rating_dates
})
invalid_rating_idx = df_ratings.sample(60, random_state=7).index
df_ratings.loc[invalid_rating_idx, "Stars"] = 8

inventory = []
inv_counter = 1
for loc_id in df_restaurants["Location_ID"]:
    sampled_items = random.sample(list(df_menu_items["Item_ID"]), min(100, NUM_MENU_ITEMS))
    for item_id in sampled_items:
        inventory.append({
            "Inventory_ID": inv_counter,
            "Location_ID": loc_id,
            "Item_ID": item_id,
            "Stock_Quantity": random.randint(0, 200),
            "Reorder_Level": random.randint(10, 50),
            "Last_Restocked_Date": fake.date_between(start_date="-30d", end_date="today")
        })
        inv_counter += 1
df_inventory = pd.DataFrame(inventory)

wastage_reasons = np.array(["Overproduction", "Spoilage", "Preparation Error", "Customer Return", "Expired Ingredients"])
wastage_days_ago = np.random.randint(0, 365, NUM_WASTAGE)
wastage_dates = [datetime.now().date() - timedelta(days=int(d)) for d in wastage_days_ago]
df_wastage = pd.DataFrame({
    "Wastage_ID": np.arange(1, NUM_WASTAGE + 1),
    "Item_ID": np.random.randint(1, NUM_MENU_ITEMS + 1, NUM_WASTAGE),
    "Location_ID": np.random.randint(1, NUM_LOCATIONS + 1, NUM_WASTAGE),
    "Wastage_Date": wastage_dates,
    "Quantity_Wasted": np.random.randint(1, 25, NUM_WASTAGE),
    "Cost_Impact": np.round(np.random.uniform(50, 2000, NUM_WASTAGE), 2),
    "Reason": np.random.choice(wastage_reasons, NUM_WASTAGE)
})
bad_wastage_idx = df_wastage.sample(40, random_state=8).index
df_wastage.loc[bad_wastage_idx, "Quantity_Wasted"] = -5

output_dir = "dineiq_dataset_1M"
os.makedirs(output_dir, exist_ok=True)

df_restaurants.to_csv(f"{output_dir}/Restaurants.csv", index=False)
df_categories.to_csv(f"{output_dir}/Menu_Categories.csv", index=False)
df_menu_items.to_csv(f"{output_dir}/Menu_Items.csv", index=False)
df_customers.to_csv(f"{output_dir}/Customers.csv", index=False)
df_promotions.to_csv(f"{output_dir}/Promotions.csv", index=False)
df_orders.to_csv(f"{output_dir}/Orders.csv", index=False)
df_order_items.to_csv(f"{output_dir}/Order_Items.csv", index=False)
df_order_items.to_parquet(f"{output_dir}/Order_Items.parquet", index=False)
df_pricing_history.to_csv(f"{output_dir}/Pricing_History.csv", index=False)
df_ratings.to_csv(f"{output_dir}/Ratings.csv", index=False)
df_inventory.to_csv(f"{output_dir}/Inventory.csv", index=False)
df_wastage.to_csv(f"{output_dir}/Wastage.csv", index=False)

print("Order_Items:", len(df_order_items))
print("Done")

Order_Items: 1000000
Done


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DineIQ-BigDataAnalytics") \
    .getOrCreate()

spark_order_items = spark.read.parquet("dineiq_dataset_1M/Order_Items.parquet")
spark_orders = spark.read.parquet("dineiq_dataset_1M/Orders.parquet")

print("Loaded successfully")

D:\Users\lenovo\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Loaded successfully


In [4]:
from pyspark.sql.functions import col

print("Negative Quantity records:", spark_order_items.filter(col("Quantity") < 0).count())
print("Duplicate Orders:", spark_orders.count() - spark_orders.dropDuplicates(["Order_ID"]).count())
print("Missing/Invalid Order Dates:", spark_orders.filter(col("Order_DateTime").isNull()).count())

Negative Quantity records: 150
Duplicate Orders: 80
Missing/Invalid Order Dates: 30


In [1]:
import os
import pandas as pd


folder = "dineiq_dataset_1M"
for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(f"{folder}/{file}")
        print(file, "->", df.shape)

ModuleNotFoundError: No module named 'pandas'

In [6]:
!pip install pyarrow

In [7]:
df_order_items = pd.read_csv("dineiq_dataset/Order_Items.csv")
df_order_items.to_parquet("dineiq_dataset/Order_Items.parquet", index=False)
print("Parquet file ban gayi")

Parquet file ban gayi


In [8]:
!java -version

'java' is not recognized as an internal or external command,
operable program or batch file.


In [10]:
!conda install -c conda-forge openjdk=17 -y

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: ...working... done

# All requested packages already installed.



In [11]:
!pip install pyspark

     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.3/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.5/450.1 MB 1.5 MB/s eta 0:05:08
     ---------------------------------------- 0.8/450.1 MB 1.1 MB/s eta 0:06:58
     ---------------------------------------- 1.6/450.1 MB 2.1 MB/s eta 0:03:31
     ---------------------------------------- 1.6/450.1 MB 2.1 MB/s eta 0:03:31
     ---------------------------------------- 2.1/450.1 MB 1.7 MB/s eta 0:04:30
     ---------------------------------------- 2.6/450.1 MB 1.8 MB/s eta 0:04:11
     ---------------------------------------- 2.6/450.1 MB 1.8 MB/s eta 0:04:11
     ---------------------------------------- 3.1/450.1 MB 1.7 MB/s eta 0:04:21
     ---------------------------------------- 3.4/450.1 MB 1.7 MB/s eta 0:04:28
     ---------------------------------------- 3.4/450.1 MB 1.

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("DineIQ-BigDataAnalytics") \
    .getOrCreate()

D:\Users\lenovo\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
spark = SparkSession.builder \
    .appName("DineIQ-BigDataAnalytics") \
    .getOrCreate()

# Parquet file ko Spark DataFrame mein load karna
spark_order_items = spark.read.parquet("dineiq_dataset_1M/Order_Items.parquet")

print("Spark Session Active & Schema:")
spark_order_items.printSchema()
spark_order_items.show(5)

Spark Session Active & Schema:
root
 |-- Order_Item_ID: long (nullable = true)
 |-- Order_ID: long (nullable = true)
 |-- Item_ID: integer (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Discount_Applied: double (nullable = true)

+-------------+--------+-------+--------+----------+----------------+
|Order_Item_ID|Order_ID|Item_ID|Quantity|Unit_Price|Discount_Applied|
+-------------+--------+-------+--------+----------+----------------+
|            1|       1|    105|       1|   1552.69|             0.0|
|            2|       1|      5|       1|   1572.78|          157.28|
|            3|       1|    134|       1|    1467.4|             0.0|
|            4|       1|     11|       1|    717.52|             0.0|
|            5|       1|     39|       1|   2424.41|             0.0|
+-------------+--------+-------+--------+----------+----------------+
only showing top 5 rows


In [6]:
# 1. Baqi CSV files ko bhi Parquet format mein save karna (SRS Big Data Storage Requirement)
import pandas as pd

df_orders_pd = pd.read_csv("dineiq_dataset_1M/Orders.csv")
df_orders_pd.to_parquet("dineiq_dataset_1M/Orders.parquet", index=False)

df_menu_pd = pd.read_csv("dineiq_dataset_1M/Menu_Items.csv")
df_menu_pd.to_parquet("dineiq_dataset_1M/Menu_Items.parquet", index=False)

df_rest_pd = pd.read_csv("dineiq_dataset_1M/Restaurants.csv")
df_rest_pd.to_parquet("dineiq_dataset_1M/Restaurants.parquet", index=False)

print("Baqi tables bhi successfully Parquet format mein convert ho gayi hain!")

# 2. In tamam Parquet files ko PySpark DataFrame mein load karna
spark_orders = spark.read.parquet("dineiq_dataset_1M/Orders.parquet")
spark_menu = spark.read.parquet("dineiq_dataset_1M/Menu_Items.parquet")
spark_restaurants = spark.read.parquet("dineiq_dataset_1M/Restaurants.parquet")

# 3. Multi-table Join using PySpark
unified_spark_df = spark_order_items \
    .join(spark_orders, "Order_ID") \
    .join(spark_menu, "Item_ID") \
    .join(spark_restaurants, "Location_ID")

print("Multi-table Spark Join Successful! Total rows:", unified_spark_df.count())

# 4. SRS Analytics: City-wise Total Revenue calculation via Spark
from pyspark.sql.functions import col, sum as spark_sum

city_revenue_spark = unified_spark_df.groupBy("City") \
    .agg(spark_sum(col("Quantity") * col("Unit_Price")).alias("Total_Revenue")) \
    .orderBy(col("Total_Revenue").desc())

print("\nCity-wise Total Revenue (Spark Processing):")
city_revenue_spark.show()

Baqi tables bhi successfully Parquet format mein convert ho gayi hain!
Multi-table Spark Join Successful! Total rows: 1001598

City-wise Total Revenue (Spark Processing):
+----------+--------------------+
|      City|       Total_Revenue|
+----------+--------------------+
|    Lahore|4.6245249140997356E8|
|Faisalabad| 3.091273537200198E8|
|    Multan|2.3320943598002797E8|
|   Karachi|2.3182610565002054E8|
| Islamabad| 2.273007358500216E8|
|Rawalpindi| 7.562079976999979E7|
+----------+--------------------+



In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Orders aur Order Items ko Pandas mein load karna (Machine Learning processing ke liye)
df_orders_pd = pd.read_csv("dineiq_dataset_1M/Orders.csv")
df_order_items_pd = pd.read_csv("dineiq_dataset_1M/Order_Items.csv")

# 2. Date column ko datetime format mein convert karna
df_orders_pd["Order_DateTime"] = pd.to_datetime(df_orders_pd["Order_DateTime"])

# 3. Customer Total Spend (Monetary) calculate karne ke liye join
df_merged_ml = df_order_items_pd.merge(df_orders_pd, on="Order_ID", how="inner")
df_merged_ml["Line_Total"] = df_merged_ml["Quantity"] * df_merged_ml["Unit_Price"]

# 4. RFM Metrics (Recency, Frequency, Monetary) create karna
snapshot_date = df_orders_pd["Order_DateTime"].max() + pd.Timedelta(days=1)

rfm_df = df_orders_pd.groupby("Customer_ID").agg({
    "Order_DateTime": lambda x: (snapshot_date - x.max()).days, # Recency
    "Order_ID": "count" # Frequency
})
rfm_df.columns = ["Recency", "Frequency"]

# Monetary value add karna
monetary_df = df_merged_ml.groupby("Customer_ID")["Line_Total"].sum().rename("Monetary")
rfm_df = rfm_df.join(monetary_df).dropna()

# 5. Feature Scaling aur KMeans Clustering apply karna
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_df)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df["Customer_Cluster"] = kmeans.fit_predict(rfm_scaled)

print("Customer Segmentation (K-Means) Successfully Completed!")
print(rfm_df.head(10))

Customer Segmentation (K-Means) Successfully Completed!
             Recency  Frequency  Monetary  Customer_Cluster
Customer_ID                                                
1               73.0          2  35590.78                 1
2              251.0          1  16741.41                 3
3               58.0          2  37578.40                 1
4               32.0          3  42709.62                 2
5               98.0          1  16181.50                 1
6               11.0          1  11427.63                 1
7              133.0          1  10562.02                 1
8              357.0          1  13439.81                 3
9              237.0          1  14703.10                 3
10              42.0          5  76798.37                 0


In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Orders aur Order Items ko Pandas mein load karna (Machine Learning processing ke liye)
df_orders_pd = pd.read_csv("dineiq_dataset_1M/Orders.csv")
df_order_items_pd = pd.read_csv("dineiq_dataset_1M/Order_Items.csv")

# 2. Date column ko datetime format mein convert karna
df_orders_pd["Order_DateTime"] = pd.to_datetime(df_orders_pd["Order_DateTime"])

# 3. Customer Total Spend (Monetary) calculate karne ke liye join
df_merged_ml = df_order_items_pd.merge(df_orders_pd, on="Order_ID", how="inner")
df_merged_ml["Line_Total"] = df_merged_ml["Quantity"] * df_merged_ml["Unit_Price"]

# 4. RFM Metrics (Recency, Frequency, Monetary) create karna
snapshot_date = df_orders_pd["Order_DateTime"].max() + pd.Timedelta(days=1)

rfm_df = df_orders_pd.groupby("Customer_ID").agg({
    "Order_DateTime": lambda x: (snapshot_date - x.max()).days, # Recency
    "Order_ID": "count" # Frequency
})
rfm_df.columns = ["Recency", "Frequency"]

# Monetary value add karna
monetary_df = df_merged_ml.groupby("Customer_ID")["Line_Total"].sum().rename("Monetary")
rfm_df = rfm_df.join(monetary_df).dropna()

# 5. Feature Scaling aur KMeans Clustering apply karna
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_df)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df["Customer_Cluster"] = kmeans.fit_predict(rfm_scaled)

print("Customer Segmentation (K-Means) Successfully Completed!")
print(rfm_df.head(10))

Customer Segmentation (K-Means) Successfully Completed!
             Recency  Frequency  Monetary  Customer_Cluster
Customer_ID                                                
1               73.0          2  35590.78                 1
2              251.0          1  16741.41                 3
3               58.0          2  37578.40                 1
4               32.0          3  42709.62                 2
5               98.0          1  16181.50                 1
6               11.0          1  11427.63                 1
7              133.0          1  10562.02                 1
8              357.0          1  13439.81                 3
9              237.0          1  14703.10                 3
10              42.0          5  76798.37                 0


In [10]:
from pyspark.sql.functions import col, year, month, dayofweek, hour

# 1. Line Total (Quantity aur Unit_Price ka product) calculate karna
unified_spark_df = unified_spark_df.withColumn(
    "Line_Total", col("Quantity") * col("Unit_Price")
)

# 2. Order_DateTime se Date-time features extract karna
unified_spark_df = unified_spark_df \
    .withColumn("Order_Year", year(col("Order_DateTime"))) \
    .withColumn("Order_Month", month(col("Order_DateTime"))) \
    .withColumn("Order_DayOfWeek", dayofweek(col("Order_DateTime"))) \
    .withColumn("Order_Hour", hour(col("Order_DateTime")))

# 3. Profit Margin feature generate karna (Unit Price aur Cost ke base par)
if "Cost" in unified_spark_df.columns and "Unit_Price" in unified_spark_df.columns:
    unified_spark_df = unified_spark_df.withColumn(
        "Profit_Margin", col("Unit_Price") - col("Cost")
    )

# Result verify karne ke liye schema aur sample rows dekhna
print("Feature Engineering Successful!")
unified_spark_df.printSchema()
unified_spark_df.select("Order_ID", "Item_ID", "Quantity", "Unit_Price", "Line_Total", "Order_Hour", "City").show(5)

Feature Engineering Successful!
root
 |-- Location_ID: long (nullable = true)
 |-- Item_ID: integer (nullable = true)
 |-- Order_ID: long (nullable = true)
 |-- Order_Item_ID: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Discount_Applied: double (nullable = true)
 |-- Customer_ID: long (nullable = true)
 |-- Order_DateTime: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Promotion_ID: double (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Item_Name: string (nullable = true)
 |-- Category_ID: long (nullable = true)
 |-- Base_Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Is_Available: long (nullable = true)
 |-- Location_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Opening_Date: string (nullable = true)
 |-- Line_Total: double (nullable = true)
 |-- Ord

Py4JJavaError: An error occurred while calling o122.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 18.0 failed 1 times, most recent failure: Lost task 0.0 in stage 18.0 (TID 18) (DESKTOP-TAB6KCU executor driver): TaskResultLost (result lost from block manager)
Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$3(DAGScheduler.scala:3318)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:3318)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:3310)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:3310)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1363)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.